In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 81 samples and 13598 columns.
First few rows of the data:


,death.time..month.,death.event,relapse.time..month.,relapse.event.1,sample_title,tissue,cell line,genotype/variation,date.of.birth,gender,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,39.000000,1,17.466667,1.0,A271,lung cancer,NaN,NaN,9/28/1941,F,...,8.709060,6.720906,7.794208,8.700092,6.659364,7.210290,9.130210,7.788665,6.820204,7.948768
1,6.533333,1,4.900000,1.0,A272,lung cancer,NaN,NaN,9/11/1947,F,...,8.536846,7.101716,8.194533,9.164981,7.237020,7.072964,8.914102,7.098715,5.951072,8.348900
2,66.700000,1,58.166667,1.0,A306,lung cancer,NaN,NaN,2/20/1948,F,...,9.092578,7.227904,8.330834,9.027126,6.730124,7.091477,8.913696,6.769134,5.779236,8.045843
3,50.133333,1,50.133333,0.0,A438,lung cancer,NaN,NaN,6/16/1938,F,...,8.090881,6.188668,7.288823,8.461460,5.855103,6.579545,8.449606,6.560251,5.589890,7.269757
4,40.966667,1,33.333333,1.0,A442,lung cancer,NaN,NaN,6/16/1945,F,...,9.091247,7.010743,8.400720,9.252639,7.285403,6.846199,9.130345,7.211239,6.716373,8.030868


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for death.time..month. or death.event: 1
Number of samples with complete data for both variables: 80
Number of censored samples (event_var == 0.0): 20
Number of events (event_var == 1.0): 60
Censored ratio (among complete): 20 / 80 = 0.250 (25.0%)
Number of samples with incomplete data for relapse.time..month. or relapse.event.1: 1
Number of samples with complete data for both variables: 80
Number of censored samples (event_var == 0.0): 42
Number of events (event_var == 1.0): 38
Censored ratio (among complete): 42 / 80 = 0.525 (52.5%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE120622/description.json
